# Self-play & population-based training — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Improving by training against evolving opponents
Self-play creates its own curriculum by making yesterday’s policy today’s opponent.

In [ ]:
# 1) return-to-go labels a trajectory for sequence modeling
rewards=np.array([1.,0.,2.,1.]); rtg=np.array([rewards[i:].sum() for i in range(len(rewards))])
plt.figure(figsize=(6,3)); plt.step(range(len(rtg)),rtg,where='mid'); plt.title('return-to-go conditions future actions')
assert np.allclose(rtg,[4,3,3,1])

In [ ]:
# 2) an action token can be predicted from state and desired return features
X=np.array([[1.,4.],[0.,3.],[1.,3.]]); w=np.array([0.7,0.2]); score=X@w
plt.figure(figsize=(6,3)); plt.bar(range(3),score); plt.title('sequence model action scores')
assert int(np.argmax(score))==0

In [ ]:
# 3) self-play ratings update after a win
ra,rb=1000.,1000.; ea=1/(1+10**((rb-ra)/400)); k=32; ra2=ra+k*(1-ea); rb2=rb+k*(0-(1-ea))
plt.figure(figsize=(6,3)); plt.bar(['A before','A after','B before','B after'],[ra,ra2,rb,rb2]); plt.title('opponent strength is part of the curriculum')
assert abs(ra2-1016)<1e-12 and abs(rb2-984)<1e-12

In [ ]:
# 4) MCTS-style visit counts become a policy target
visits=np.array([10,30,60],dtype=float); pi=visits/visits.sum()
plt.figure(figsize=(6,3)); plt.bar(range(3),pi); plt.title('search visits supervise the policy')
assert abs(pi[2]-0.6)<1e-12

In [ ]:
# 5) value and policy combine into a search score
prior=np.array([0.2,0.5,0.3]); value=np.array([0.7,0.4,0.6]); score=value+0.5*prior
plt.figure(figsize=(6,3)); plt.bar(range(3),score); plt.title('Alpha-style search balances prior and value')
assert int(np.argmax(score))==0